# Experiment 3: Replicate with DeepSeek

## Setup

In [6]:
from nest_asyncio import apply
from os import environ
from llama_index.llms.replicate import Replicate

apply()
# environ['REPLICATE_API_TOKEN'] = ''

llm_replicate = \
    Replicate(
        model='meta/meta-llama-3-70b-instruct', 
        api_token=environ['REPLICATE_API_TOKEN'],
    )

## Load documents

In [7]:
from llama_index.embeddings.huggingface import HuggingFaceEmbedding
from llama_index.core import Settings, SimpleDirectoryReader

embed_model = HuggingFaceEmbedding(model_name='BAAI/bge-small-en-v1.5')

Settings.llm = llm_replicate
Settings.embed_model = embed_model

overview_statement_pdf = SimpleDirectoryReader(input_files=['../data/overview_statement.pdf']).load_data()
task_breakdown_pdf = SimpleDirectoryReader(input_files=['../data/task_breakdown.pdf']).load_data()

## Interpret documents

In [8]:
from llama_index.core import VectorStoreIndex

overview_statement = \
    VectorStoreIndex \
        .from_documents(overview_statement_pdf) \
        .as_query_engine(similarity_top_k=3) \
        .query('Tell me about the project.')

task_breakdown = \
    VectorStoreIndex \
        .from_documents(task_breakdown_pdf) \
        .as_query_engine(similarity_top_k=3) \
        .query('List all project tasks.')

print(overview_statement, task_breakdown)



Based on the provided context information, the project is about creating a Conversational AI Assistant application for Chicago WideCast Smart-Home Services, a startup company. The goal is to automate all business process workflows using generative AI technologies to serve customers and employees online. The Conversational AI Assistant will facilitate various tasks, including customer support, order management, and technical support, among others, for the company's different roles, including managers, account specialists, technical support specialists, and customers. 

Based on the provided context information, the list of project tasks is:

1. Write Plan
2. Review Plan
3. Preparation for review
4. Review Meeting
5. Rework
6. Write Risk Mitigation and Contingency Plan
7. Review Risk Mitigation and Contingency Plan
8. Preparation for review
9. Review Meeting
10. Rework
11. Write requirements
12. Write Use Case Model
13. Review Requirements/Use Case Model
14. Preparation for review
15. 

## Start chat

In [9]:
from llama_index.core.llms import ChatMessage

def any_message(role, sections):
    message = f"Ask the {role} to predict the amount of effort required to complete sections:\n\n"
    for _, section in enumerate(sections):
        message += f'- {section}\n'
    return message

print(
    llm_replicate.chat(
        [
            ChatMessage(
                role='system', 
                content= \
                    'Consider the overview statement:\n\n. ' + \
                    f"{overview_statement}:\n\n." + \
                    'The tasks for the software project are:\n\n' + \
                    f"{task_breakdown}:\n\n." + \
                    'Consider the overview statement document of the project Chicago WideCase Smart-Home Services. ' + \
                    'Given the work listed in the task description document\n\n.' + \
                    '- Tag each task with a unique ID that starts with `REQ-001`.\n' + \
                    '- Create effort estimate for each task.',
            ),
            ChatMessage(
                role='user', 
                content= \
                    any_message(
                        'Project Manager',
                        ['Project plan', 'Risk Mitigation and Contingency Plan'],
                    ),
            ),
            ChatMessage(
                role='user', 
                content= \
                    any_message(
                        'Requirement Engineer',
                        ['Requirement'],
                    ),
            ),
            ChatMessage(
                role='user', 
                content= \
                    any_message(
                        'System Engineer',
                        ['Analysis', 'Design'],
                    ),
            ),
            ChatMessage(
                role='user', 
                content= \
                    any_message(
                        'Test Engineer',
                        ['Testing'],
                    ),
            ),
            ChatMessage(
                role='user', 
                content= \
                    any_message(
                        'Documentation Engineer',
                        ['Documentation'],
                    ),
            ),
            ChatMessage(
                role='user', 
                content= 'Summarize the effort estimation.',
            ),
        ],
    ),
)

assistant: 

Here is the task list with unique IDs, effort estimates, and predictions from the respective team members:

**Project Plan**

1. REQ-001: Write Plan - 4 hours (predicted by Project Manager)
2. REQ-002: Review Plan - 2 hours (predicted by Project Manager)
3. REQ-003: Preparation for review - 1 hour (predicted by Project Manager)
4. REQ-004: Review Meeting - 1 hour (predicted by Project Manager)
5. REQ-005: Rework - 2 hours (predicted by Project Manager)

**Risk Mitigation and Contingency Plan**

6. REQ-006: Write Risk Mitigation and Contingency Plan - 8 hours (predicted by Project Manager)
7. REQ-007: Review Risk Mitigation and Contingency Plan - 4 hours (predicted by Project Manager)
8. REQ-008: Preparation for review - 1 hour (predicted by Project Manager)
9. REQ-009: Review Meeting - 1 hour (predicted by Project Manager)
10. REQ-010: Rework - 4 hours (predicted by Project Manager)

**Requirements**

11. REQ-011: Write requirements - 16 hours (predicted by Requirement Eng